# Preparing a crop for annotation in spatiato

Crops the whole-slide store of `macroseg.baseline` (image, segmentations, tables) and keeps what the
annotator needs: all image channels and the `membrane_dapi` segmentation with its table.
The crop is written to `macroseg_runs`; open it on the Mac with napari + spatiato.


In [ ]:
from pathlib import Path

from spatialdata import bounding_box_query, read_zarr

from macroseg.config import load_config

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
cfg = load_config(ROOT / "configs" / "exp201_mouse_neuroblastoma.yaml")
sdata = read_zarr(cfg.zarr)
sdata

In [ ]:
Y0, X0 = 3000, 2000   # top-left corner, in pixels of the whole slide
SIZE = 2048           # side in pixels (~350 µm at 0.17 µm/pixel)

crop = bounding_box_query(
    sdata,
    axes=("y", "x"),
    min_coordinate=[Y0, X0],
    max_coordinate=[Y0 + SIZE, X0 + SIZE],
    target_coordinate_system="global",
    filter_table=True,
)
crop

In [ ]:
crop = crop.subset(["macsima_image", "membrane_dapi", "membrane_dapi_intensities"])
crop

In [ ]:
OUT = Path("/srv/data/massimoz/macroseg_runs/exp201_crop.zarr")
crop.write(OUT, overwrite=True)